In [5]:
from models.TransBTS.TransBTS_downsample8x_skipconnection import TransBTS
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import nibabel as nib
import pandas as pd
from monai.transforms import LoadImage, Compose, Spacing, Resize, EnsureChannelFirst, ScaleIntensity
import numpy as np
import torch

In [6]:
class TransBTS_model_dataset_enoder_only(Dataset):
    def __init__(self, BrainMRIFilesPath, CSVPath, transform=None):        
        self.BrainMRIFilesPath = BrainMRIFilesPath
        self.CSVPath = CSVPath
        self.transform = transform
        self.df = pd.read_excel(self.CSVPath , engine='openpyxl')
        
        self.Files = os.listdir(self.BrainMRIFilesPath)

        self.ID_To_Filename = {}
        self.IDs = []
        self.Lables = {}
        for file in self.Files:
            strc = file.split('-')
            self.ID_To_Filename[int(strc[0][-3:])] = file
            self.IDs.append(int(strc[0][-3:]))
            self.Lables[int(strc[0][-3:])] = np.random.randint(0, 2, size=(4096, 512), dtype=np.int32)

    def __len__(self):
        return len(self.IDs)

    def __getitem__(self, index):
        ID = self.IDs[index]
        filename = self.ID_To_Filename[ID]
        file_path = os.path.join(self.BrainMRIFilesPath ,filename)
        if self.transform is not None:
            image = transforms(file_path)              # returns torch.Tensor
            array = image.numpy()[0]                    
        
        age = self.df[self.df['IXI_ID'] == ID]['AGE'].values[0]
        Label = self.Lables[ID]
        return array, Label
        

In [10]:
transforms = Compose([
    LoadImage(image_only=True),
    EnsureChannelFirst(),                   # shape: (1, H, W, D)
    ScaleIntensity(),                       # optional: normalize to 0-1
    Spacing(pixdim=(1.0, 1.0, 1.0), mode="bilinear"),  # resample to 1mm spacing
    Resize(spatial_size=(128, 128, 128), mode='trilinear')  # resize to 128³
])

In [18]:
BrainMRIFilesPath = r"C:\Users\Rishabh\Documents\PreprocessedData1mm"
CSVPath = os.path.join(r'C:\Users\Rishabh\Documents\TransBTS', 'IXI.xlsx')
ds = TransBTS_model_dataset_enoder_only(BrainMRIFilesPath, CSVPath , transforms)
loader = DataLoader(ds, batch_size=1, shuffle=True, num_workers=0)

In [21]:
_, model = TransBTS(_conv_repr=True, _pe_type="learned")
model.eval()

BTS(
  (linear_encoding): Linear(in_features=128, out_features=512, bias=True)
  (position_encoding): LearnedPositionalEncoding()
  (pe_dropout): Dropout(p=0.1, inplace=False)
  (transformer): TransformerModel(
    (net): IntermediateSequential(
      (0): Residual(
        (fn): PreNormDrop(
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (fn): SelfAttention(
            (qkv): Linear(in_features=512, out_features=1536, bias=False)
            (attn_drop): Dropout(p=0.1, inplace=False)
            (proj): Linear(in_features=512, out_features=512, bias=True)
            (proj_drop): Dropout(p=0.1, inplace=False)
          )
        )
      )
      (1): Residual(
        (fn): PreNorm(
          (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (fn): FeedForward(
            (net): Sequential(
              (0): Linear(in_features=512, out_features=4096, bias=True)
              (1

In [28]:
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()

In [30]:
from tqdm import tqdm
num_epochs = 2
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    with tqdm(loader, desc=f"Epoch {epoch+1}/{num_epochs}") as t:
        for batch in t:
            Imagevol = batch[0]
            Imagevol = Imagevol.unsqueeze(1)
            Label = batch[1]
            encoder_output, intmd_encoder_outputs = model(Imagevol)  
            encoder_output = torch.sigmoid(encoder_output)
            loss = criterion(encoder_output, Label.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            t.set_postfix(loss=loss.item())

            
            print(loss)

Epoch 1/2:   6%|███▌                                                        | 1/17 [00:18<05:03, 18.96s/it, loss=0.802]

tensor(0.8022, grad_fn=<BinaryCrossEntropyBackward0>)


Epoch 1/2:  12%|███████                                                     | 2/17 [00:37<04:39, 18.65s/it, loss=0.792]

tensor(0.7919, grad_fn=<BinaryCrossEntropyBackward0>)


Epoch 1/2:  18%|██████████▌                                                 | 3/17 [00:55<04:20, 18.61s/it, loss=0.782]

tensor(0.7819, grad_fn=<BinaryCrossEntropyBackward0>)


Epoch 1/2:  18%|██████████▌                                                 | 3/17 [01:02<04:53, 20.94s/it, loss=0.782]


KeyboardInterrupt: 